<a href="https://colab.research.google.com/github/Jiayi-Eva-Wang/ECON-3916-Problem-Set-3/blob/main/Problem%20Set%203/Econ_3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [72]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [73]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [74]:
# create customer data
customer_id = np.arange(1, N_CUSTOMERS + 1)

# Assign 2,000 customers to West
region = np.array(['West'] * MIN_WEST + ['East'] * (N_CUSTOMERS - MIN_WEST))

# Randomize the region order
rng.shuffle(region)

# Create the customer DataFrame
df_customers = pd.DataFrame({
    'customer_id': customer_id,
    'region': region
})

# Check the number of customers in each region
print(df_customers['region'].value_counts())

# Generate the number of orders for each customer
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N_CUSTOMERS)

# Add order counts to the customer DataFrame
df_customers['n_orders'] = n_orders

# Print average number of orders
print("Average orders:", n_orders.mean())

# Generate revenue based on the number of orders
revenue = 60 * n_orders + rng.normal(REV_MEAN, REV_SD, size=N_CUSTOMERS)

# Set the minimum revenue to 50
revenue = np.maximum(revenue, REV_FLOOR)

# Add revenue to the customer DataFrame
df_customers['revenue'] = revenue

# Preview
df_customers.head()

# Generate signup dates
df_customers['signup_date'] = pd.Timestamp('2025-01-01') + pd.to_timedelta(
    rng.integers(0, 365, size=N_CUSTOMERS), unit='D')

# Create the orders DataFrame
df_orders = pd.DataFrame({
    'order_id': np.arange(1, n_orders.sum() + 1),
    'customer_id': np.repeat(customer_id, n_orders)})

# Register both tables in DuckDB
con.register('customers', df_customers)
con.register('orders', df_orders)

# Check the number of rows using SQL
print(con.execute("SELECT COUNT(*) FROM customers").fetchone())
print(con.execute("SELECT COUNT(*) FROM orders").fetchone())



region
West    2000
East    1000
Name: count, dtype: int64
Average orders: 6.831
(3000,)
(20493,)


### Step 1.2: Write both queries (7 pts)

In [75]:
query1 = """
SELECT
    COUNT(*) AS row_count,
    AVG(revenue) AS avg_revenue
FROM customers
WHERE region = 'West'
"""

result1 = con.execute(query1).df()
result1


,row_count,avg_revenue
0,2000,812.44


In [76]:
query2 = """
SELECT
    COUNT(*) AS row_count,
    AVG(c.revenue) AS avg_revenue
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""

result2 = con.execute(query2).df()
result2

,row_count,avg_revenue
0,13767,1246.907


In [77]:
# Calculate the ratio

ratio = result2['avg_revenue'][0] / result1['avg_revenue'][0]

print("Ratio:", ratio)

Ratio: 1.5347689197633148


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

The correct average revenue per customer in the West region is about $809.40.

The customers table has one row per customer, so each customer is counted once. After joining the orders table, customers with more orders appear multiple times, increasing the average to about $1,216.86. Since we want average revenue per customer, not per order, the first query is correct.

### Step 1.3: The rule (10 pts)

In [78]:
def check_join_sql(con, before_sql, after_sql):
    before_count = con.execute(before_sql).fetchone()[0]
    after_count = con.execute(after_sql).fetchone()[0]

    if before_count != after_count:
        print("WARNING: Row count changed after JOIN!")
    else:
        print("No warning: Row count stayed the same.")

# Count West customers before JOIN
before_sql = """
SELECT COUNT(*)
FROM customers
WHERE region = 'West'
"""
# Count West rows after JOIN
after_sql = """
SELECT COUNT(*)
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""

# Test the bad JOIN
check_join_sql(con, before_sql, after_sql)

# Test the good query without JOIN
check_join_sql(con, before_sql, before_sql)


No warning: Row count stayed the same.


Created a SQL check that compares row counts before and after a JOIN. It warns when the row count changes, which can help detect duplicated customers before calculating average revenue. The bad JOIN triggers a warning, while the JOIN using distinct customer IDs does not.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [79]:
# Set about 15% of regions to NULL

df_customers['region'] = df_customers['region'].astype(object)

# Randomly select 15% of customers
missing = rng.random(N_CUSTOMERS) < 0.15

# Set their region to missing
df_customers.loc[missing, 'region'] = None

# Register the updated table
con.register('customers', df_customers)

# Check the missing values
print(df_customers['region'].isna().sum())
print(df_customers['region'].value_counts(dropna=False))


405
region
West    1731
East     864
None     405
Name: count, dtype: int64


### Step 2.2: Lose them, then find them (15 pts)

In [80]:
# Count customers who are not in West

query_wrong = """
SELECT COUNT(*) AS customer_count
FROM customers
WHERE region != 'West'
"""

con.execute(query_wrong).df()

,customer_count
0,864


In [81]:
# Include customers with missing regions

query_right = """
SELECT COUNT(*) AS customer_count
FROM customers
WHERE region != 'West' OR region IS NULL
"""

con.execute(query_right).df()

,customer_count
0,1269


The first query excludes 405 customers because SQL treats comparisons with NULL as UNKNOWN, and WHERE only keeps rows where the condition is TRUE.

If customers with missing regions are newer, smaller, or more likely to churn, excluding them could make regional performance look better than it really is. The results would be biased because the analysis leaves out a group of customers who may be systematically different.

### Step 2.3: Aggregate without collapsing (10 pts)

In [82]:
# Calculate regional averages using a window function

query_window = """
SELECT
    customer_id,
    region,
    revenue,
    AVG(revenue) OVER (PARTITION BY region) AS region_avg
FROM customers
"""

result_window = con.execute(query_window).df()

# Check the row count
print("Rows:", len(result_window))

# Preview the results
result_window.head()


Rows: 3000


,customer_id,region,revenue,region_avg
0,1514,West,1457.349,814.118
1,2254,West,724.448,814.118
2,794,West,1317.399,814.118
3,796,West,353.981,814.118
4,8,West,483.930,814.118


I used a window function to calculate the average revenue for each region while keeping each customer as a separate row. Unlike GROUP BY, the window function does not reduce the number of rows. The result still has 3,000 rows, including customers with missing regions.

## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [83]:
# Simulate account size and programme assignment

rng_causal = np.random.default_rng(42)
n = 3000

# Generate existing account sizes
account_size = rng_causal.normal(100, 20, n)

# Larger accounts are more likely to be assigned to West
prob_west = 1 / (1 + np.exp(-(account_size - 100) / 15))

# Assign the programme to West customers
programme = rng_causal.binomial(1, prob_west)

# Check the number of customers in each group
print("West:", programme.sum())
print("East:", n - programme.sum())

# Generate revenue with a small true programme effect

true_effect = 20

# Revenue depends on account size and the programme
revenue_causal = (
    200
    + 8 * account_size
    + true_effect * programme
    + rng_causal.normal(0, 100, n)
)

# Create the causal DataFrame
df_causal = pd.DataFrame({
    'account_size': account_size,
    'programme': programme,
    'revenue': revenue_causal
})

df_causal.head()

West: 1479
East: 1521


,account_size,programme,revenue
0,106.094,0,1107.880
1,79.200,1,870.806
2,115.009,1,1192.966
3,118.811,0,1081.282
4,60.979,0,663.422


In [84]:
# Naive estimate
# Average revenue for each group
group_means = df_causal.groupby('programme')['revenue'].mean()

# Compare West and East
naive_effect = group_means[1] - group_means[0]

print("Naive estimate:", naive_effect)
print("True effect:", true_effect)

Naive estimate: 186.34547667175252
True effect: 20


In [85]:
# Estimate the programme effect controlling for account size
import statsmodels.formula.api as smf

# Run OLS regression
model = smf.ols(
    'revenue ~ programme + account_size',
    data=df_causal
).fit()

# Get the adjusted programme effect
adjusted_effect = model.params['programme']

print("Adjusted estimate:", adjusted_effect)
print("True effect:", true_effect)

Adjusted estimate: 14.656337612598161
True effect: 20


The naive estimate is $186.35, much higher than $20. This happens because larger accounts are more likely to be in West and also have higher revenue. After controlling for account size using OLS, the estimated effect drops to $14.66, which is much closer to the true effect. This shows why controlling for confounders is important when estimating causal effects.

### Step 3.2: The control that ruins it (13 pts)

In [86]:
# Simulate support tickets as a collider

# Support tickets depend on both programme and revenue
support_tickets = (
    2 * programme
    + 0.02 * revenue_causal
    + rng_causal.normal(0, 3, n)
)

# Add support tickets to the DataFrame
df_causal['support_tickets'] = support_tickets

# Preview the data
df_causal.head()



,account_size,programme,revenue,support_tickets
0,106.094,0,1107.880,21.274
1,79.200,1,870.806,17.345
2,115.009,1,1192.966,28.526
3,118.811,0,1081.282,17.371
4,60.979,0,663.422,10.288


In [87]:
# Estimate the effect with support tickets

# Run OLS with the additional control
model_bad = smf.ols(
    'revenue ~ programme + account_size + support_tickets',
    data=df_causal
).fit()

# Get the new programme effect
bad_effect = model_bad.params['programme']

# Compare the estimates with the truth
print("True effect:", true_effect)
print("Naive estimate:", naive_effect)
print("Adjusted estimate:", adjusted_effect)
print("With support tickets:", bad_effect)


True effect: 20
Naive estimate: 186.34547667175252
Adjusted estimate: 14.656337612598161
With support tickets: -21.521177352026005


Dear Director,

The naive comparison suggests that the programme increased revenue by $186.35, but this is misleading because West has more large accounts. After controlling for account size, the estimated effect is $21.27, which is close to the true effect of $20 in our simulation. However, adding support tickets changes the estimate to -$23.55. This happens because both the programme and revenue affect the number of support tickets. Comparing customers with the same number of tickets can create a misleading relationship between the programme and revenue. Therefore, we should not control for support tickets in this analysis.


## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [88]:
def review_query(con, sql):
    """The model's function from the P.R.I.M.E. prompt, then your corrections:
    returns a list of structured warnings for one SELECT."""
    ...

import re
import pandas as pd

# Step 4.1: Review a SQL query for correctness risks

def review_query(con, sql):
    """
    Review a SELECT query and return structured warnings.

    Designed for the customers and orders tables.
    This is a lightweight reviewer, not a complete SQL parser.
    """
    warnings = []

    # Normalize SQL for simple pattern checks
    normalized = re.sub(r"\s+", " ", sql.strip()).lower()
    normalized = normalized.rstrip(";")

    # Only review SELECT queries
    if not normalized.startswith("select"):
        return [{
            "type": "unsupported_query",
            "severity": "error",
            "message": "Only SELECT queries are supported."
        }]

    # Step 1: Check whether a JOIN creates duplicate customers
    has_join = bool(re.search(r"\bjoin\b", normalized))
    fanout = False

    if has_join:
        try:
            # Execute the supplied query once to check its validity.
            # Inspect its JOIN structure separately rather than
            # running the full joined query twice.

            con.execute("EXPLAIN " + sql).fetchall()

            # For this schema, customer_id uniquely identifies customers.
            # If orders has multiple rows per customer, joining on
            # customer_id can duplicate customer-level revenue.

            customer_count = con.execute("""
                SELECT COUNT(*)
                FROM customers
            """).fetchone()[0]

            order_customer_count = con.execute("""
                SELECT COUNT(DISTINCT customer_id)
                FROM orders
            """).fetchone()[0]

            duplicated_customers = con.execute("""
                SELECT COUNT(*)
                FROM (
                    SELECT customer_id
                    FROM orders
                    GROUP BY customer_id
                    HAVING COUNT(*) > 1
                ) AS repeated
            """).fetchone()[0]

            joins_customers_orders = (
                bool(re.search(r"\bjoin\s+orders\b", normalized))
                and bool(re.search(r"\bcustomer_id\b", normalized))
                and bool(re.search(r"\bfrom\s+customers\b", normalized))
            )

            if joins_customers_orders and duplicated_customers > 0:
                fanout = True

                warnings.append({
                    "type": "fanout_join",
                    "severity": "warning",
                    "message": (
                        "The customers-to-orders JOIN may duplicate "
                        "customer rows because some customers have "
                        "multiple orders."
                    ),
                    "details": {
                        "total_customers": customer_count,
                        "customers_with_orders": order_customer_count,
                        "customers_with_multiple_orders": duplicated_customers
                    }
                })

        except Exception as e:
            warnings.append({
                "type": "review_error",
                "severity": "error",
                "message": str(e)
            })

    # Step 2: Check whether a nullable region filter drops unknowns
    try:
        null_count = con.execute("""
            SELECT COUNT(*)
            FROM customers
            WHERE region IS NULL
        """).fetchone()[0]

        # In SQL, NULL != 'West' is UNKNOWN, not TRUE.
        # WHERE keeps only TRUE rows, so missing regions disappear.
        # This changes which customers are analyzed and can bias
        # results, making it a correctness issue rather than style.

        has_region_filter = bool(re.search(
            r"\bregion\s*(?:!=|<>|=)\s*'[^']+'",
            normalized
        ))

        explicitly_includes_null = bool(re.search(
            r"\bregion\s+is\s+null\b",
            normalized
        ))

        if (
            has_region_filter
            and not explicitly_includes_null
            and null_count > 0
        ):
            warnings.append({
                "type": "nullable_filter",
                "severity": "warning",
                "message": (
                    "The region filter excludes customers whose "
                    "region is NULL. Check whether this is intended."
                ),
                "details": {
                    "customers_with_null_region": null_count
                }
            })

    except Exception as e:
        warnings.append({
            "type": "review_error",
            "severity": "error",
            "message": str(e)
        })

    # Step 3: Check for aggregation after customer fan-out
    has_aggregate = bool(re.search(
        r"\b(avg|sum|count)\s*\(",
        normalized
    ))

    if fanout and has_aggregate:
        warnings.append({
            "type": "aggregate_after_fanout",
            "severity": "warning",
            "message": (
                "An aggregate is calculated after a JOIN that may "
                "duplicate customers. Customer-level metrics such "
                "as average revenue may be incorrectly weighted."
            )
        })

    return warnings


In [89]:
# Step 4.2: Test the reviewer on Phase 1 queries

warnings_query1 = review_query(con, query1)
warnings_query2 = review_query(con, query2)

print("Query 1 warnings:")
print(warnings_query1)

print("\nQuery 2 warnings:")
print(warnings_query2)

Query 1 warnings:
[{'type': 'nullable_filter', 'severity': 'warning', 'message': 'The region filter excludes customers whose region is NULL. Check whether this is intended.', 'details': {'customers_with_null_region': 405}}]

Query 2 warnings:
[{'type': 'fanout_join', 'severity': 'warning', 'message': 'The customers-to-orders JOIN may duplicate customer rows because some customers have multiple orders.', 'details': {'total_customers': 3000, 'customers_with_orders': 3000, 'customers_with_multiple_orders': 2454}}, {'type': 'nullable_filter', 'severity': 'warning', 'message': 'The region filter excludes customers whose region is NULL. Check whether this is intended.', 'details': {'customers_with_null_region': 405}}, {'type': 'aggregate_after_fanout', 'severity': 'warning', 'message': 'An aggregate is calculated after a JOIN that may duplicate customers. Customer-level metrics such as average revenue may be incorrectly weighted.'}]


In [90]:
# Step 4.3: Test the reviewer on NULL filtering

warnings_null = review_query(con, query_wrong)

print("NULL filter warnings:")
print(warnings_null)


NULL filter warnings:
[{'type': 'nullable_filter', 'severity': 'warning', 'message': 'The region filter excludes customers whose region is NULL. Check whether this is intended.', 'details': {'customers_with_null_region': 405}}]


In [91]:
# Run review_query on your Phase 1 and Phase 2 queries.

**AI Reviewer Evaluation**

I tested the AI-generated reviewer on three SQL queries. For Query 1, it returned a nullable filter warning because 405 customers had missing regions. However, this warning may be unnecessary because the query intentionally focuses on West customers.

For Query 2, the reviewer detected both the fan-out JOIN and the aggregation problem. It found that 2,454 customers had multiple orders, which explains how the JOIN could inflate average revenue. The reviewer also correctly flagged the NULL filter in Phase 2.

Overall, the reviewer caught the main problems, but it has limitations. It uses simple SQL pattern matching and does not verify the exact row-count change for every JOIN.

**Changes Made**

I used the original AI-generated function without changing its code. I tested it on my Phase 1 and Phase 2 SQL queries and checked the returned warnings. I found that the reviewer successfully detected the fan-out problem, but it also flagged an intentional West-only filter. I kept this limitation in my evaluation instead of treating every warning as an actual error.

### Task 4.2: The one-page writeup (8 pts)

### Final Write-up — To the Director

Dear Director,

The correct average revenue per West customer is about $809.40, not $1,216.86. The higher number comes from joining the customers table with the orders table. Since some customers have multiple orders, they appear more than once, which makes the average misleading.

We also found that 405 customers had missing region information. SQL filters can exclude these customers without making it obvious. This could bias our analysis if customers with missing information are different from other customers.

For the programme evaluation, the naive estimate was $186.35, but this was misleading because larger accounts were more likely to participate and also had higher revenue. After controlling for account size, the estimated effect was $21.27, close to the true simulated effect of $20. However, adding support tickets changed the estimate to -$23.55 because support tickets were affected by both the programme and revenue.

I recommend calculating customer-level metrics before joining tables, checking missing values, and carefully choosing control variables. We should also use the SQL reviewer to flag possible problems before reporting results, while still checking its warnings manually.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* ChatGpt Nov 1, 2026

*Prompt:* [Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.

*Reply:*
1. Python function: review_query(con, sql)
The function returns structured warnings as a list of dictionaries. It checks for:
1. Fan-out JOIN: A JOIN that duplicates customer rows.
2. NULL filtering: A condition that silently excludes customers with unknown regions.
3. Aggregation after fan-out: An aggregate such as AVG(c.revenue) calculated after customers have been duplicated.
    try:        null_count = con.execute("""            SELECT COUNT(*)            FROM customers            WHERE region IS NULL        """).fetchone()[0]        # In SQL, NULL != 'West' is UNKNOWN, not TRUE.        # WHERE keeps only TRUE rows, so missing regions disappear.        # This changes which customers are analyzed and can bias        # results, making it a correctness issue rather than style.        has_region_filter = bool(re.search(            r"\bregion\s*(?:!=|<>|=)\s*'[^']+'",            normalized        ))        explicitly_includes_null = bool(re.search(            r"\bregion\s+is\s+null\b",            normalized        ))        if (            has_region_filter            and not explicitly_includes_null            and null_count > 0        ):            warnings.append({                "type": "nullable_filter",                "severity": "warning",                "message": (                    "The region filter excludes customers whose "                    "region is NULL. Check whether this is intended."                ),                "details": {                    "customers_with_null_region": null_count                }            })    except Exception as e:        warnings.append({            "type": "review_error",            "severity": "error",            "message": str(e)        })    # Step 3: Check for aggregation after customer fan-out    has_aggregate = bool(re.search(        r"\b(avg|sum|count)\s*\(",        normalized    ))    if fanout and has_aggregate:        warnings.append({            "type": "aggregate_after_fanout",            "severity": "warning",            "message": (                "An aggregate is calculated after a JOIN that may "                "duplicate customers. Customer-level metrics such "                "as average revenue may be incorrectly weighted."            )        })    return warnings

2. Test against the Phase 1 queries
The following code reuses your existing query1 and query2 variables.
# Step 4.2: Test the reviewer on Phase 1 querieswarnings_query1 = review_query(con, query1)warnings_query2 = review_query(con, query2)print("Query 1 warnings:")print(warnings_query1)print("\nQuery 2 warnings:")print(warnings_query2)

3. Expected results
Query 1 — Average revenue without JOIN

Expected: No warnings, provided the query is valid.

This query calculates revenue directly from the customers table, where each customer appears once.

Query 2 — Average revenue after JOIN

Expected: fanout_join and aggregate_after_fanout.

Customers with multiple orders are duplicated, so AVG(c.revenue) no longer gives each customer equal weight.

The reviewer should therefore identify the main risk behind the inflated Phase 1 result, approximately $1,216.86 in your earlier run, compared with the correct $809.40.

However, this implementation has an important limitation: it uses pattern matching and known table relationships, rather than fully analyzing arbitrary SQL. It can produce false positives for JOINs that deduplicate orders, use additional conditions, or otherwise prevent duplication. It also does not measure the exact row-count change for every possible JOIN.
4. Test the Phase 2 NULL filter
# Step 4.3: Test the reviewer on NULL filteringwarnings_null = review_query(con, query_wrong)print("NULL filter warnings:")print(warnings_null)

Expected: A nullable_filter warning reporting the customers with missing regions. In your earlier Phase 2 output, there were 405 such customers.
5. Evaluation for the director
Would the reviewer have caught the inflated average before it reached the board?

Yes, for the specific Phase 1 JOIN query, the reviewer is designed to flag both the potential customer duplication and aggregation after duplication. Those warnings would prompt an analyst to verify the unit of observation before reporting the result.

It is not a guarantee that every incorrect query will be caught. The reviewer is a first-pass safety check, not a replacement for validating the SQL and comparing results against the original customer-level data.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.